In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 6: the Gale surface-temperature map

T = T_base + f, with T_base from `thermal1d` (all-blocks parameters) and f = `flann_safe_no_ti` refit on all fitting rows. Design: `docs/plan.md`. All logic is in `map_export.py`; this notebook only drives it.

- **`MODE = "dry"`**: a 100 x 110 window around the rover (11,000 pixels, 2 chunks; encoded in all four formats and checked) and a sparse sample of every 20th pixel across the crater (baseline and residual only; for the temperature range, the colour scale and per-column time over all terrain). Output in `outputs/maps/ground/dryrun/`.
- **`MODE = "full"`**: the whole 718 x 738 grid, output in `outputs/maps/ground/`. About 3-4 h; restarts skip finished chunks.

The 5 Ls values run in parallel, one process each (`map_export.run_all_ls`). Progress goes to `_working/map_progress_ls{LLL}.log` in each run folder; the float32 arrays and the static grid are in `_working/` too.

In [ ]:
# --- config ---
import json
import subprocess
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t
import residual as r
import map_export as m

MODE = "full"                 # "dry" or "full"
EXPECTED_TAU = {15.0: 0.78, 75.0: 0.49, 195.0: 0.69, 255.0: 1.09, 340.0: 1.11}   # plan.md, 2 decimals
EXPECTED_PARAMS = (0.8193, 0.0312, 0.7396)                                        # plan.md, 4 decimals
ROVER_200M = (152, 186)       # rover pixel on the 200 m grid (row 304, col 372 at 100 m)
REPRO_TOL_K = 0.05            # map code path vs the Step 4 model at the rover
CLAMP_BASELINE_TOL_K = 0.05   # refit clamp range of baseline_k vs flann_safe_clamp_ranges.csv
DRY_RUNS = {
    "rover": ((102, 202, 1, 131, 241, 1), True),    # 100 x 110 pixels centred on the rover; exported
    "sample": ((5, 718, 20, 5, 738, 20), False),    # every 20th pixel across the crater; not exported
}
FULL_RUNS = {"full": (m.FULL_WINDOW, True)}
CHUNK_COLS = m.CHUNK_COLS
FEATURES = paths.DATA_DIR / "gale_dust_devil_features_100m.csv"
META = paths.DATA_DIR / "gale_dust_devil_meta.json"
TABLE = paths.DATA_DIR / "gale_rems_training.csv"
FIT_PARAMS = paths.OUT_DIR / "fit_params.csv"
CLAMP_CSV = paths.RESIDUAL_OUT_DIR / "flann_safe_clamp_ranges.csv"

assert MODE in ("dry", "full"), MODE
RUNS = DRY_RUNS if MODE == "dry" else FULL_RUNS
OUT = paths.MAP_OUT_DIR / "dryrun" if MODE == "dry" else paths.MAP_OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
run_dir = {name: OUT / name if MODE == "dry" else OUT for name in RUNS}
print("mode:", MODE, "| out:", OUT, "| runs:", {k: v[0] for k, v in RUNS.items()}, "| chunk:", CHUNK_COLS)

## Inputs: tau per Ls, parameters, static 200 m grid

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(TABLE))
tau_tab = m.map_tau(tbl)
print(tau_tab.round(3).to_string())
for ls, tau in EXPECTED_TAU.items():
    assert round(tau_tab.loc[ls, "tau"], 2) == tau, f"tau at Ls {ls}: {tau_tab.loc[ls, 'tau']:.3f}, plan says {tau}"
LS_TAU = [(ls, float(tau_tab.loc[ls, "tau"])) for ls in m.MAP_LS_DEG]

fits = pd.read_csv(FIT_PARAMS).set_index("fold")
params = tuple(fits.loc[m.FIT_PARAMS_LABEL, list(t.PARAM_NAMES)].to_numpy(dtype=float))
assert np.allclose(params, EXPECTED_PARAMS, atol=5e-5), f"parameters {params}"
assert not fits.loc[m.FIT_PARAMS_LABEL, [f"{p}_at_bound" for p in t.PARAM_NAMES]].any(), "all-blocks fit on a bound"
print("baseline parameters (g_diff, f_ir, ti_scale):", np.round(params, 4))

meta = m.load_meta(META)
t0 = time.time()
static = m.build_static_grid(FEATURES, meta)
STATIC_NPZ = OUT / m.WORK_SUBDIR / "static_200m.npz"
STATIC_NPZ.parent.mkdir(parents=True, exist_ok=True)
np.savez(STATIC_NPZ, **static)
print(f"static grid in {time.time() - t0:.0f} s, saved to {STATIC_NPZ.name}")
transform_full = m.map_transform(meta)
rover_lonlat = (meta["rover_lon"], meta["rover_lat"])
assert m.lonlat_to_rowcol(transform_full, *rover_lonlat) == ROVER_200M, "rover not at the expected 200 m pixel"
cov = ["slope_deg", "aspect_deg", "ti_themis_tiu", "albedo_mean"]
print("200 m pixel", ROVER_200M, {c: round(float(static[c][ROVER_200M]), 3) for c in cov})
print("rover's 100 m pixels over the training sols:")
print(tbl.groupby("sol")[cov].first().agg(["min", "median", "max"]).round(3).to_string())
v = static["valid"]
print("crater (valid 200 m pixels): TI percentiles 1/50/99 =", np.round(np.percentile(static["ti_themis_tiu"][v], [1, 50, 99])),
      "| slope 1/50/99 =", np.round(np.percentile(static["slope_deg"][v], [1, 50, 99]), 1))

## Residual model: `flann_safe_no_ti` refit on all fitting rows

In [ ]:
t0 = time.time()
fit_frame, fit_target, sols, Ts_step4 = m.residual_training_rows(tbl, params)
model = m.MapResidual(fit_frame, fit_target)
print(f"{m.RESIDUAL_VARIANT}: {len(fit_frame)} fitting rows, {model.pipe[-1].coef_.size} basis columns, "
      f"static terms {model.static}, in {time.time() - t0:.0f} s")
ref = pd.read_csv(CLAMP_CSV, index_col=0)
for c, (lo, hi) in model.ranges.items():
    tol = CLAMP_BASELINE_TOL_K if c == "baseline_k" else 1e-9
    assert abs(lo - ref.loc["lo", c]) <= tol and abs(hi - ref.loc["hi", c]) <= tol, f"clamp range of {c} differs from Step 5"
print("clamp ranges (match Step 5's all-blocks ranges):", {c: tuple(round(float(x), 3) for x in v) for c, v in model.ranges.items()})
in_sample = model.predict(fit_frame)[0]
print(f"in-sample RMSE: baseline {np.sqrt(np.mean(fit_target ** 2)):.2f} K, baseline + residual "
      f"{np.sqrt(np.mean((fit_target - in_sample) ** 2)):.2f} K (in-sample; the skill score is the LOBO 4.86 K)")

## Check: the map code path reproduces the Step 4 model at the rover

For the training sol nearest each map Ls, `map_export.run_chunk` runs that sol's own rover inputs (100 m pixel, its Ls and tau) as a one-column batch. The Step 4 model is the same solver run over all sols as one batch (`Ts_step4`). Pass: within 0.05 K at all 48 frames.

In [ ]:
usable = sols[~sols.tau_is_placeholder & ~sols.sol.between(*m.SUSPECT_SOLS)]
repro = []
for ls in m.MAP_LS_DEG:
    i = int(np.abs(np.mod(usable.ls_deg - ls + 180.0, 360.0) - 180.0).idxmin())
    s = sols.loc[i]
    inp = {"lat_deg": t.lat_from_row(np.array([s.row])), "slope_deg": np.array([s.slope_deg]),
           "aspect_deg": np.array([s.aspect_deg]), "albedo_mean": np.array([s.albedo_mean]),
           "ti_themis_tiu": np.array([s.ti_themis_tiu])}
    frames, _ = m.run_chunk(inp, s.ls_deg, s.tau, params)
    diff = np.abs(frames[0].astype(np.float64) - Ts_step4[i, m.FRAME_STEPS])
    repro.append({"map_ls": ls, "sol": int(s.sol), "sol_ls": s.ls_deg, "tau": s.tau, "max_abs_diff_k": diff.max()})
repro = pd.DataFrame(repro)
print(repro.round(4).to_string(index=False))
assert repro.max_abs_diff_k.max() < REPRO_TOL_K, "map code path does not reproduce Step 4"
print(f"PASS: max |map - Step 4| = {repro.max_abs_diff_k.max():.4f} K (< {REPRO_TOL_K} K)")

## Baseline: the 5 Ls in parallel

In [ ]:
summaries = {}
for name, (window, _) in RUNS.items():
    print(f"run {name}: window {window}, started {time.strftime('%H:%M:%S')}")
    t0 = time.time()
    res = m.run_all_ls(LS_TAU, params, STATIC_NPZ, run_dir[name], window, CHUNK_COLS)
    summaries[name] = pd.DataFrame(res).set_index("ls_deg")
    print(f"  wall time {time.time() - t0:.0f} s")
    print(summaries[name].round(2).to_string())
n_full = int(static["valid"].sum())
for name, s in summaries.items():
    if s.cols_run.min() > 0:
        print(f"{name}: {s.ms_per_col.mean():.2f} ms/col per worker with 5 in parallel -> full grid ({n_full} pixels) "
              f"about {n_full * s.ms_per_col.max() / 1e3 / 3600:.1f} h wall time; peak memory per worker "
              f"{s.peak_memory_mb.max():.0f} MB")

## Residual, master arrays and exports

In [ ]:
results = {}
for name, (window, export) in RUNS.items():
    d = run_dir[name]
    inp, valid = m.window_inputs(static, window)
    transform = m.map_transform(meta, window)
    rows = []
    for ls, tau in LS_TAU:
        tag = m.ls_tag(ls)
        base = np.load(d / m.WORK_SUBDIR / "master" / f"{tag}_base.npy")
        Tflat, clamped = m.add_residual(base[:, valid], inp, ls, tau, model)
        T = np.full_like(base, np.nan)
        T[:, valid] = Tflat
        np.save(d / m.WORK_SUBDIR / "master" / f"{tag}.npy", T)
        resid = Tflat - base[:, valid]
        lo, hi = m.COLOR_RANGE_K
        row = {"ls_deg": ls, "tau": tau, "T_min_k": Tflat.min(), "T_p1_k": np.percentile(Tflat, 1),
               "T_p99_k": np.percentile(Tflat, 99), "T_max_k": Tflat.max(),
               "resid_min_k": resid.min(), "resid_max_k": resid.max(),
               "color_below_frac": float(np.mean(Tflat < lo)), "color_above_frac": float(np.mean(Tflat > hi)),
               **{f"clamped_{c}": f for c, f in clamped.items()}}
        if export:
            files = m.export_ls(T, tag, d, transform)
            rr, cc = ROVER_200M[0] - window[0], ROVER_200M[1] - window[3]
            assert window[2] == window[5] == 1, "exports are checked on unstrided windows only"
            row.update(m.check_exports(T, files, transform, rover_lonlat, (rr, cc)))
            row.update({f"{f}_mb": m.folder_size_mb(p) for f, p in files.items()})
        rows.append(row)
    results[name] = pd.DataFrame(rows).set_index("ls_deg")
    if export:
        m.write_legend(d)
    pd.options.display.float_format = "{:.3f}".format
    print(f"run {name}:")
    print(results[name].T.to_string())
    results[name].to_csv(d / "map_summary.csv")

## Manifest

In [ ]:
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT, check=True).stdout.strip()
dirty = bool(subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=ROOT, check=True).stdout.strip())
from rasterio.crs import CRS
for name, (window, export) in RUNS.items():
    if not export:
        continue
    d = run_dir[name]
    res = results[name]
    shape = [int(len(range(*window[0:3]))), int(len(range(*window[3:6])))]
    manifest = {
        "title": "Gale Crater surface (ground) temperature, physics baseline + residual model",
        "created": time.strftime("%Y-%m-%d %H:%M:%S"), "git_commit": commit, "git_dirty": dirty,
        "mode": MODE, "window_200m": list(window),
        "grid": {"shape_rows_cols": shape, "pixel_size_note": "about 200 m: 2 x 2 block means of the 100 m feature table",
                 "row0": "north edge", "transform_gdal": list(m.map_transform(meta, window)),
                 "transform_note": "lon = c + a * col, lat = f + e * row, at pixel corners (degrees)",
                 "dropped": "last 100 m column (1477 is odd)", "crs_authority": m.CRS_AUTHORITY,
                 "crs_wkt": CRS.from_user_input(m.CRS_AUTHORITY).to_wkt(),
                 "nodata_rule": "a 200 m pixel is nodata if any of its 4 100 m pixels is absent from the table"},
        "frames": {"ltst_hours": m.FRAME_HOURS.tolist(), "kind": "instants: the solver step ending at each LTST",
                   "labels": [m.frame_label(h) for h in m.FRAME_HOURS]},
        "scenarios": [{"tag": m.ls_tag(ls), "ls_deg": ls, "tau": tau,
                       "tau_rule": f"median per-sol tau of training sols within +-{m.TAU_WINDOW_DEG:g} deg Ls, placeholder sols excluded",
                       "n_sols": int(tau_tab.loc[ls, "n_sols"]), "blocks": tau_tab.loc[ls, "blocks"]} for ls, tau in LS_TAU],
        "encodings": {
            "A": {"files": "cog/gale_tsurf_ls{LLL}.tif", "dtype": "int16", "offset_k": m.COG_OFFSET_K, "scale_k": m.COG_SCALE_K,
                  "nodata": m.COG_NODATA, "bands": "one per frame, description 'LTST hh:mm'", "compression": "DEFLATE, predictor 2"},
            "B": {"files": "png16/ls{LLL}/ltst_{hhmm}.png", "dtype": "uint16 grayscale", "offset_k": m.U16_OFFSET_K,
                  "scale_k": m.U16_SCALE_K, "nodata": m.U16_NODATA},
            "C": {"files": "png_color/ls{LLL}/ltst_{hhmm}.png", "legend": "png_color/legend.png", "colormap": m.COLOR_MAP,
                  "range_k": list(m.COLOR_RANGE_K), "levels": m.COLOR_LEVELS, "nodata": "transparent",
                  "clipped_fraction_below_above": {m.ls_tag(ls): [float(res.loc[ls, "color_below_frac"]), float(res.loc[ls, "color_above_frac"])] for ls, _ in LS_TAU}},
            "D": {"files": "raw/gale_tsurf_ls{LLL}.u16", "dtype": "uint16 little-endian", "order": "C, shape (frame, row, col)",
                  "shape": [m.N_FRAMES] + shape, "offset_k": m.U16_OFFSET_K, "scale_k": m.U16_SCALE_K, "nodata": m.U16_NODATA},
            "decode": "T_K = offset_k + scale_k * value, except value == nodata"},
        "model": {"baseline": "thermal1d 1D soil conduction, all-blocks fit", "params": dict(zip(t.PARAM_NAMES, map(float, params))),
                  "fixed": {"eps": t.EPS, "b_tau": t.B_TAU, "rho_c": t.RHO_C},
                  "residual_variant": m.RESIDUAL_VARIANT, "residual_static_terms": model.static,
                  "clamp_ranges": {c: [float(lo), float(hi)] for c, (lo, hi) in model.ranges.items()},
                  "clamped_fraction_of_pixel_frames": {m.ls_tag(ls): {c: float(res.loc[ls, f"clamped_{c}"]) for c in model.ranges} for ls, _ in LS_TAU},
                  "clamp_note": "ti_themis_tiu and albedo_mean are clamped but are not terms of flann_safe_no_ti",
                  "lobo_macro_rmse_k": 4.86, "benchmark_lobo_macro_rmse_k": 5.38},
        "caveats": [
            "Spatial skill is not measured: all rover positions lie within about 10 x 5 km.",
            "About 82% of crater pixels have THEMIS TI below the rover's range (361 tiu); there the baseline runs at ti_scale x THEMIS TI, untested by the rover data.",
            "Slope is clamped to the rover's 1.25-12.27 deg in the residual only; steep walls and Mount Sharp's flanks rest on the physics baseline.",
            "One tau per Ls for the whole map; no CO2 frost, no atmospheric coupling, homogeneous soil.",
        ],
    }
    (d / "map_manifest.json").write_text(json.dumps(manifest, indent=1))
    print("wrote", d / "map_manifest.json")

## Preview

In [ ]:
for name, (window, _) in RUNS.items():
    d = run_dir[name]
    fig, axes = plt.subplots(2, len(LS_TAU), figsize=(3.2 * len(LS_TAU), 6.2), squeeze=False)
    for k, (ls, tau) in enumerate(LS_TAU):
        T = np.load(d / m.WORK_SUBDIR / "master" / f"{m.ls_tag(ls)}.npy")
        for i, hour in enumerate((14.0, 2.0)):
            j = int(np.flatnonzero(m.FRAME_HOURS == hour)[0])
            im = axes[i, k].imshow(T[j], cmap=m.COLOR_MAP, vmin=m.COLOR_RANGE_K[0], vmax=m.COLOR_RANGE_K[1])
            axes[i, k].set_title(f"Ls {ls:g}, tau {tau:.2f}, {m.frame_label(hour)}", fontsize=8)
            axes[i, k].set_xticks([]); axes[i, k].set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.6, label="K")
    fig.savefig(d / "preview.png", dpi=110)
    plt.close(fig)
    print("saved", d / "preview.png")